# Лаборатория «Интуиция ML» — Регрессия

## Линейная регрессия: от первых принципов до sklearn

> **Это рекомендуемая ТОЧКА ВХОДА в лабораторную.** Порядок чтения: **этот ноутбук** → `01_classification_intuition` → `03_ensembles_intuition`.
> Здесь градиентный спуск разбирается в самой простой форме: MSE — это идеальный параболоид с одним минимумом, нет нелинейностей и вероятностей. После этого ноутбука переход к логистической регрессии в 01 будет очевидным — там добавляется только сигмоида (нелинейная squash-функция) и log-loss вместо MSE, всё остальное идентично.

Здесь — **линейная регрессия**: предсказание непрерывной величины. Преимущества для интуиции:

- функция ошибки — **MSE**, идеальный параболоид с одним минимумом;
- $\partial \mathcal{L} / \partial w_j \propto x_j$ — это объединяющая формула (та же, что и для логрегрессии в 01);
- **подвох с немасштабированными данными**: на этом примере GD сразу разлетится без скейлинга, и мы вживую это покажем;
- для линейной регрессии есть **аналитическое решение** $w = (X^TX)^{-1}X^T y$ — минимум можно найти за один шаг матричной алгебры, без итераций.

Задача: по площади квартиры и расстоянию до центра предсказать её цену.

## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from data_generators import make_apartment_dataset, APARTMENT_TRUE_COEFS
from plotting import (
    plot_loss_surface_3d,
    plot_loss_contour,
    plot_loss_curve,
    plot_weight_evolution,
)

plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")
np.random.seed(42)

## Часть 1. Данные

Сгенерируем 200 квартир. Истинная зависимость:
$$
\text{price} = 50\,000 + 1500 \cdot \text{area} - 800 \cdot \text{dist} + \varepsilon
$$
где $\varepsilon \sim \mathcal{N}(0, 5000)$ — шум. **Истинные коэффициенты $(1500, -800, 50000)$ нам известны** — обучение должно их восстановить.

In [ ]:
df = make_apartment_dataset(n=200, noise_scale=5000, random_state=42)
print(df.head())
print(f'\nИстинные коэффициенты: {APARTMENT_TRUE_COEFS}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['area'], df['price'], s=20, alpha=0.7)
axes[0].set_xlabel('площадь, м²'); axes[0].set_ylabel('цена')
axes[0].set_title('Цена vs площадь — растёт')

axes[1].scatter(df['dist'], df['price'], s=20, alpha=0.7, color='C1')
axes[1].set_xlabel('расстояние от центра, км'); axes[1].set_ylabel('цена')
axes[1].set_title('Цена vs расстояние — падает')

plt.tight_layout()
plt.show()

In [ ]:
fig = go.Figure(go.Scatter3d(
    x=df['area'], y=df['dist'], z=df['price'],
    mode='markers',
    marker=dict(size=3, color=df['price'], colorscale='Viridis', showscale=True),
    name='данные'
))
fig.update_layout(
    title='Цена квартиры в пространстве (площадь, расстояние)',
    scene=dict(xaxis_title='площадь, м²', yaxis_title='расстояние, км', zaxis_title='цена'),
    width=800, height=600,
)
fig.show()

## Часть 2. Модель и функция ошибки

Модель: $\hat{y} = w_1 \cdot \text{area} + w_2 \cdot \text{dist} + b$. Хотим найти такие $(w_1, w_2, b)$, чтобы предсказания были близки к истинным ценам.

### MSE — Mean Squared Error

$$
\mathcal{L} = \frac{1}{n}\sum_{i=1}^n (\hat{y}_i - y_i)^2.
$$

**Почему именно квадрат?**
- большие промахи штрафуются непропорционально сильнее (квадратично),
- функция гладкая (везде дифференцируемая),
- получается **выпуклая** loss-поверхность с одним минимумом — GD гарантированно его найдёт.

In [ ]:
X = df[['area', 'dist']].to_numpy()
y = df['price'].to_numpy()

def mse_loss(X, y, w, b):
    y_hat = X @ w + b
    return np.mean((y_hat - y) ** 2)

### 3D-визуализация loss(w1, w2)

Зафиксируем $b$ в его оптимуме (для визуализации). Видим **идеальный параболоид** — outline формы $aw_1^2 + bw_2^2 + \dots$. Один минимум, выпуклая функция, можно бесконечно скатываться.

In [ ]:
# найдём оптимальное b через нормальное уравнение для нашего удобства
_lr = LinearRegression().fit(X, y)
b_fixed = _lr.intercept_
print(f'Оптимальное b (по нормальному уравнению) = {b_fixed:.2f}')
print(f'Оптимальные w = {_lr.coef_}')

def loss_surface(w1, w2):
    return mse_loss(X, y, np.array([w1, w2]), b_fixed)

# центрируем диапазон вокруг оптимума, по w1 берём узкий диапазон, по w2 — шире
w1_opt, w2_opt = _lr.coef_
w1_range = np.linspace(w1_opt - 200, w1_opt + 200, 50)
w2_range = np.linspace(w2_opt - 1500, w2_opt + 1500, 50)

fig = plot_loss_surface_3d(loss_surface, w1_range, w2_range,
                           title='MSE как функция (w1, w2) — выпуклый параболоид',
                           w1_name='w1 (площадь)', w2_name='w2 (расстояние)')
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_loss_contour(loss_surface, w1_range, w2_range, ax=ax,
                  title='Контурная карта MSE — концентрические эллипсы',
                  w1_name='w1 (площадь)', w2_name='w2 (расстояние)')
plt.show()

**Контуры — концентрические эллипсы**, причём эллипс вытянут по $w_2$ (расстояние). Почему именно так — увидим через секунду через формулу градиента.

## Часть 3. Градиент MSE — та же логика, что и в logreg

Снова цепное правило для одного примера ($\ell = (\hat{y} - y)^2$):

$$
\frac{\partial \ell}{\partial w_j} = 2(\hat{y} - y) \cdot \frac{\partial \hat{y}}{\partial w_j} = 2(\hat{y} - y) \cdot x_j.
$$

Усреднение по выборке:

$$
\boxed{\frac{\partial \mathcal{L}}{\partial w_j} = \frac{2}{n}\sum_{i=1}^n (\hat{y}_i - y_i) \cdot x_{ij}.}
$$

**Это та же самая формула**, что и для логистической регрессии (с точностью до коэффициента 2). $x_j$ снова в множителе — поэтому градиент по $w_1$ (вес при площади) больше, чем градиент по $w_2$ (вес при расстоянии): площадь измеряется десятками, а расстояние — единицами.

Это объясняет вытянутость эллипса на контурной карте: чтобы изменить loss на ту же величину, по $w_1$ нужно сдвинуться на маленькое расстояние, а по $w_2$ — на большое.

## Часть 4. Реализация на NumPy

In [ ]:
class LinRegNumpy:
    # Линейная регрессия с обучением через градиентный спуск.

    def __init__(self, lr: float = 1e-5, n_epochs: int = 1000):
        self.lr = lr
        self.n_epochs = n_epochs
        self.w = None
        self.b = None
        self.history = {'loss': [], 'w': [], 'b': []}

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.history = {'loss': [], 'w': [], 'b': []}

        for epoch in range(self.n_epochs):
            y_hat = X @ self.w + self.b

            loss = mse_loss(X, y, self.w, self.b)
            self.history['loss'].append(loss)
            self.history['w'].append(self.w.copy())
            self.history['b'].append(self.b)

            err = y_hat - y
            grad_w = (2 / n) * (X.T @ err)
            grad_b = (2 / n) * err.sum()

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            # защита от расходимости — если loss взорвался, дальше считать бессмысленно
            if not np.isfinite(loss):
                print(f'  ⚠ расходимость на эпохе {epoch}, loss = {loss}')
                break
        return self

    def predict(self, X):
        return X @ self.w + self.b

## Часть 5. Обучение БЕЗ скейлинга — наглядная катастрофа

На немасштабированных данных:
- `area` ~ десятки, `dist` ~ единицы, `price` ~ десятки тысяч;
- ошибка $(\hat{y} - y)$ на старте — порядка 100\,000;
- градиент $(\hat{y} - y) \cdot x_j$ — порядка $10^6$–$10^7$;
- с обычным lr=0.01 веса прыгают на $10^4$ за шаг и улетают.

**Покажем катастрофу вживую:**

In [ ]:
print('Попытка 1: lr = 1e-3 (катастрофа ожидается)')
model_explode = LinRegNumpy(lr=1e-3, n_epochs=20).fit(X, y)
print(f'  финальный loss: {model_explode.history["loss"][-1]:.2e}')
print(f'  первые 5 loss: {[f"{l:.2e}" for l in model_explode.history["loss"][:5]]}')

**Видно:** loss мгновенно становится `inf` или растёт как лавина. Это и есть тот случай, когда `lr` слишком велик для масштаба градиентов.

### Решение №1: уменьшить lr вручную

In [ ]:
print('Попытка 2: lr = 1e-7 — крошечный, но устойчивый')
model_unscaled = LinRegNumpy(lr=1e-7, n_epochs=10000).fit(X, y)
print(f'  финальный loss: {model_unscaled.history["loss"][-1]:.4f}')
print(f'  финальные w = {model_unscaled.w}, b = {model_unscaled.b:.2f}')
print(f'  истинные:    w = [{APARTMENT_TRUE_COEFS["w_area"]}, {APARTMENT_TRUE_COEFS["w_dist"]}], b = {APARTMENT_TRUE_COEFS["intercept"]}')

Смотри: `w_area ≈ 1500` (истинное), `w_dist` пока далеко от истинного `-800`, и `b` вообще близок к нулю. **Модель сходится, но КРАЙНЕ медленно** — за 10000 эпох не дошла. На немасштабированных данных это типичная картина.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_loss_curve(model_unscaled.history, ax=axes[0],
                title='Loss по эпохам (без скейлинга, lr=1e-7)', log_y=True)
plot_weight_evolution(model_unscaled.history, ax=axes[1],
                      title='Эволюция весов', weight_names=['w1 (площадь)', 'w2 (расстояние)'])
plt.tight_layout()
plt.show()

## Часть 6. Решение №2: масштабирование

`StandardScaler` приводит `area` и `dist` к среднему 0 и std 1. Целевую переменную тоже отмасштабируем (для регрессии это не обязательно, но делает loss-поверхность ещё более «учебной»).

С масштабированными данными `lr=0.01` отлично работает и сходимся за десятки эпох.

In [ ]:
scaler_X = StandardScaler().fit(X)
X_scaled = scaler_X.transform(X)

# y оставим в исходных единицах для интерпретируемости b и весов
model_scaled = LinRegNumpy(lr=0.01, n_epochs=300).fit(X_scaled, y)
print(f'Со скейлингом X (lr=0.01, 300 эпох):')
print(f'  финальный loss: {model_scaled.history["loss"][-1]:.2f}')
print(f'  финальные w = {model_scaled.w}, b = {model_scaled.b:.2f}')

### Side-by-side: контурные карты loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Слева — без скейлинга
W_unsc = np.array(model_unscaled.history['w'])
w1_grid = np.linspace(W_unsc[:,0].min()-50, max(W_unsc[:,0].max()+50, 1700), 60)
w2_grid = np.linspace(W_unsc[:,1].min()-50, W_unsc[:,1].max()+200, 60)
b_unsc = model_unscaled.b
def loss_unsc(w1, w2):
    return mse_loss(X, y, np.array([w1, w2]), b_unsc)
plot_loss_contour(loss_unsc, w1_grid, w2_grid, trajectory=W_unsc, ax=axes[0],
                  title='БЕЗ скейлинга — оооочень вытянутый эллипс\n(GD ползёт черепахой)',
                  w1_name='w1 (площадь)', w2_name='w2 (расстояние)')

# Справа — со скейлингом
W_sc = np.array(model_scaled.history['w'])
margin = max(abs(W_sc[:,0]).max(), abs(W_sc[:,1]).max()) * 0.3
w1_grid = np.linspace(W_sc[:,0].min()-margin, W_sc[:,0].max()+margin, 60)
w2_grid = np.linspace(W_sc[:,1].min()-margin, W_sc[:,1].max()+margin, 60)
b_sc = model_scaled.b
def loss_sc(w1, w2):
    return mse_loss(X_scaled, y, np.array([w1, w2]), b_sc)
plot_loss_contour(loss_sc, w1_grid, w2_grid, trajectory=W_sc, ax=axes[1],
                  title='СО скейлингом — почти круг\n(GD идёт по прямой)',
                  w1_name='w1', w2_name='w2')

plt.tight_layout()
plt.show()

**Это весь смысл скейлинга в одной картинке.** Слева — годами ползём по овраг, справа — сразу к минимуму.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model_unscaled.history['loss'][:1000], label='БЕЗ скейлинга, lr=1e-7', linewidth=2)
ax.plot(model_scaled.history['loss'], label='СО скейлингом, lr=0.01', linewidth=2)
ax.set_xlabel('эпоха'); ax.set_ylabel('loss')
ax.set_title('Сходимость: со скейлингом сильно быстрее')
ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

### Восстановили ли истинные коэффициенты?

Веса в масштабированном пространстве не равны истинным $(1500, -800)$ — они в **другой системе координат**. Чтобы сравнить, переведём обратно в исходные единицы:

$$
w_j^{\text{orig}} = \frac{w_j^{\text{scaled}}}{\sigma_j}, \qquad
b^{\text{orig}} = b^{\text{scaled}} - \sum_j \frac{w_j^{\text{scaled}} \cdot \mu_j}{\sigma_j}.
$$

In [ ]:
mu = scaler_X.mean_
sigma = scaler_X.scale_
w_back = model_scaled.w / sigma
b_back = model_scaled.b - np.sum(model_scaled.w * mu / sigma)

print('Истинные коэффициенты (по которым генерили):')
print(f'  w_area = {APARTMENT_TRUE_COEFS["w_area"]}, w_dist = {APARTMENT_TRUE_COEFS["w_dist"]}, b = {APARTMENT_TRUE_COEFS["intercept"]}')
print()
print('Восстановленные (масштабированные веса -> обратное преобразование):')
print(f'  w_area = {w_back[0]:.2f}, w_dist = {w_back[1]:.2f}, b = {b_back:.2f}')
print()
print('Совпадают с точностью до шума в данных. Отлично — метод работает.')

## Часть 7. Эффект learning rate (на масштабированных данных)

In [ ]:
lrs = [0.001, 0.05, 0.5]
labels = [f'lr={lr}' for lr in lrs]
models_by_lr = [LinRegNumpy(lr=lr, n_epochs=100).fit(X_scaled, y) for lr in lrs]

fig, ax = plt.subplots(figsize=(8, 4))
for m, lbl in zip(models_by_lr, labels):
    losses = m.history['loss']
    losses = [l if np.isfinite(l) else None for l in losses]
    ax.plot(losses, label=lbl, linewidth=2)
ax.set_xlabel('эпоха'); ax.set_ylabel('loss (log)')
ax.set_title('Loss по эпохам для разных lr')
ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

for m, lbl in zip(models_by_lr, labels):
    print(f'  {lbl}: финальный loss = {m.history["loss"][-1]:.2f}')

- **lr=0.001:** очень медленно, не дошли;
- **lr=0.05:** идеально;
- **lr=0.5:** перепрыгиваем минимум, оказываемся хуже старта (или вообще расходимся).

## Часть 8. Сравнение с sklearn + аналитическое решение

### sklearn

In [ ]:
sk = LinearRegression().fit(X, y)
print(f'sklearn LinearRegression:')
print(f'  w = {sk.coef_}, b = {sk.intercept_:.2f}')
print(f'  R² на train = {sk.score(X, y):.4f}')

print(f'\nНаш numpy (со скейлингом, переведённый обратно):')
print(f'  w = {w_back}, b = {b_back:.2f}')
print(f'\nДолжны почти совпасть.')

### Аналитическое решение — нормальное уравнение

Для линейной регрессии минимум MSE можно найти **за один шаг матричной алгебры**, без всякого GD:

$$
\hat{w} = (X^TX)^{-1}X^Ty.
$$

Это и есть ответ на твой вопрос «как мы знаем, что в этой точке минимум?». Для линейной регрессии минимум **выводится формулой**. GD — лишь итеративный способ к нему прийти, который пригодится, когда формулы нет (как в нейросетях).

In [ ]:
# добавим столбец единиц для bias и решим нормальное уравнение
X_aug = np.column_stack([X, np.ones(len(X))])
theta = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
print(f'Аналитическое решение (нормальное уравнение):')
print(f'  w = [{theta[0]:.2f}, {theta[1]:.2f}], b = {theta[2]:.2f}')
print(f'\nЭто ровно sklearn.LinearRegression — sklearn по умолчанию её и решает,')
print(f'просто чуть стабильнее (через SVD, а не явный inverse).')

## Часть 9. Интерпретация коэффициентов

### Веса в исходных единицах
$$
\text{price} \approx 50000 + 1500 \cdot \text{area} - 800 \cdot \text{dist}.
$$
- $w_{\text{area}} = 1500$: каждый дополнительный м² **прибавляет 1500** к цене;
- $w_{\text{dist}} = -800$: каждый дополнительный км от центра **отнимает 800**;
- $b = 50000$: «базовая» стоимость гипотетической квартиры с `area=0` и `dist=0` (для интерполяции внутри обучающей области, экстраполяция за её пределы — нефизична).

### После скейлинга — интерпретация в std-единицах

In [ ]:
print(f'Скейлер:')
print(f'  μ_area = {mu[0]:.2f}, σ_area = {sigma[0]:.2f}')
print(f'  μ_dist = {mu[1]:.2f}, σ_dist = {sigma[1]:.2f}')
print()
print(f'Веса в масштабированном пространстве: {model_scaled.w}')
print()
print(f'Что это значит:')
print(f'  +1 std площади (~{sigma[0]:.0f} м²) → +{model_scaled.w[0]:.0f} ₽ в цене.')
print(f'  +1 std расстояния (~{sigma[1]:.0f} км) → {model_scaled.w[1]:.0f} ₽ в цене.')
print(f'  Проверка: w_scaled = w_orig · σ → {APARTMENT_TRUE_COEFS["w_area"]} · {sigma[0]:.2f} = {APARTMENT_TRUE_COEFS["w_area"]*sigma[0]:.0f} ✓')

**Вывод:** скейлинг не «ломает» интерпретацию, он меняет **единицу измерения признака** — с «метров квадратных» на «стандартных отклонений». Если ты хочешь интерпретируемые в исходных единицах коэффициенты — обратно преобразуй веса по формуле выше; если тебе важно **сравнить силу влияния разных признаков между собой** (например, что важнее — площадь или расстояние), удобнее смотреть на масштабированные веса (они в одинаковых единицах std).

## Чеклист «понял ли я»

1. Почему MSE — выпуклая функция? *(сумма квадратов линейных функций — квадратичная форма с положительно полуопределённой матрицей)*
2. Почему градиент MSE по $w_j$ снова пропорционален $x_j$? *(цепное правило, $\partial \hat{y}/\partial w_j = x_j$)*
3. Что произойдёт с GD на немасштабированных данных, если взять «обычный» lr? *(градиенты огромны, веса прыгают, loss расходится)*
4. Зачем нужен скейлинг, если можно просто уменьшить lr? *(маленький lr → очень медленная сходимость по форме «оврага»; скейлинг исправляет геометрию loss-поверхности целиком)*
5. Как восстановить интерпретацию весов в исходных единицах после скейлинга? *(делить на $\sigma_j$, поправить $b$)*
6. Почему для линейной регрессии можно обойтись без GD? *(есть аналитическое решение $w = (X^TX)^{-1}X^Ty$)*
7. Почему тогда GD вообще нужен? *(для нелинейных моделей — нейросетей, где формулы нет)*

Дальше → ноутбук **`01_classification_intuition.ipynb`**: разберём логистическую регрессию по той же схеме. Все идеи градиентного спуска, скейлинга, lr — те же; добавляется только сигмоида и log-loss.